### Parte 1 — Datamart analítico y ETL

Este notebook toma la capa `data/raw` generada por `00_generacion_datos.ipynb`, limpia el ruido controlado y produce la capa final `data/processed`.

El objetivo es dejar listas las tablas dimensionales y de hechos para Power BI, conservando el esquema estrella del proyecto AndesMarket.

Flujo:

1. Cargar tablas raw.
2. Reportar calidad inicial y ruido detectado.
3. Estandarizar tipos, fechas y textos.
4. Imputar nulos controlados.
5. Eliminar duplicados de grano.
6. Validar integridad referencial.
7. Validar y reconstruir métricas derivadas.
8. Exportar tablas limpias a `data/processed`.
9. Exportar diccionario de datos.

**Grano de la tabla de hechos:** una fila = una línea de venta (`id_venta` + `numero_linea`).

#### Paso 1: Imports, rutas y configuración

In [34]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd


# Detecta raíz del proyecto ejecutando desde notebooks/ o desde la raíz.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_RAW = DATA_DIR / "raw"
DATA_PROCESSED = DATA_DIR / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"

for carpeta in [DATA_RAW, DATA_PROCESSED, DOCS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Permite importar scripts del proyecto si luego se necesitan.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEP = ";"
ENC = "utf-8"

TABLAS = {
    "dim_cliente": "dim_cliente.csv",
    "dim_producto": "dim_producto.csv",
    "dim_tienda": "dim_tienda.csv",
    "dim_promocion": "dim_promocion.csv",
    "dim_tiempo": "dim_tiempo.csv",
    "fact_ventas": "fact_ventas.csv",
}

print("Raíz del proyecto:", PROJECT_ROOT)
print("Entrada raw:", DATA_RAW)
print("Salida processed:", DATA_PROCESSED)
print("Documentación:", DOCS_DIR)

Raíz del proyecto: c:\Users\karlo\Desktop\Proy Final Inteligencia de Negocios
Entrada raw: c:\Users\karlo\Desktop\Proy Final Inteligencia de Negocios\data\raw
Salida processed: c:\Users\karlo\Desktop\Proy Final Inteligencia de Negocios\data\processed
Documentación: c:\Users\karlo\Desktop\Proy Final Inteligencia de Negocios\docs


#### Paso 2: Funciones base de carga y reporte

In [35]:
def ruta_relativa(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def cargar_tabla(nombre: str, carpeta: Path = DATA_RAW) -> pd.DataFrame:
    ruta = carpeta / TABLAS[nombre]

    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró {ruta_relativa(ruta)}")

    df = pd.read_csv(ruta, sep=SEP, encoding=ENC)
    df.columns = df.columns.astype(str).str.strip()

    print(f"Cargado {nombre} desde {ruta_relativa(ruta)}: {len(df):,} filas x {df.shape[1]} columnas")
    return df


def reporte_calidad(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    filas = len(df)
    nulos = df.isna().sum()
    pct_nulos = (nulos / filas * 100).round(2) if filas else pd.Series(0, index=df.columns)

    resumen = pd.DataFrame({
        "columna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "nulos": nulos.values,
        "pct_nulos": pct_nulos.values,
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
    })

    print(f"\n=== Calidad: {nombre} ({filas:,} filas) ===")
    display(resumen)

    return resumen


def contar_duplicados_grano(df: pd.DataFrame, columnas: list[str]) -> int:
    return int(df.duplicated(subset=columnas).sum())


def contar_huerfanas(fact_col: pd.Series, dim_col: pd.Series) -> int:
    valores_fact = set(fact_col.dropna())
    valores_dim = set(dim_col.dropna())
    return len(valores_fact - valores_dim)


def mostrar_resumen_tablas(tablas: dict[str, pd.DataFrame], titulo: str) -> pd.DataFrame:
    resumen = pd.DataFrame([
        {
            "tabla": nombre,
            "filas": len(df),
            "columnas": len(df.columns),
        }
        for nombre, df in tablas.items()
    ])

    print(f"\n=== {titulo} ===")
    display(resumen)

    return resumen

#### Paso 3: Cargar tablas desde data/raw

In [36]:
raw = {nombre: cargar_tabla(nombre, DATA_RAW) for nombre in TABLAS}

resumen_raw = mostrar_resumen_tablas(raw, "Tablas RAW cargadas")

calidad_antes = {}
for nombre, df in raw.items():
    calidad_antes[nombre] = reporte_calidad(df, f"{nombre} raw")

Cargado dim_cliente desde data\raw\dim_cliente.csv: 5,000 filas x 8 columnas
Cargado dim_producto desde data\raw\dim_producto.csv: 506 filas x 11 columnas
Cargado dim_tienda desde data\raw\dim_tienda.csv: 15 filas x 7 columnas
Cargado dim_promocion desde data\raw\dim_promocion.csv: 29 filas x 7 columnas
Cargado dim_tiempo desde data\raw\dim_tiempo.csv: 731 filas x 10 columnas
Cargado fact_ventas desde data\raw\fact_ventas.csv: 56,193 filas x 14 columnas

=== Tablas RAW cargadas ===


,tabla,filas,columnas
0,dim_cliente,5000,8
1,dim_producto,506,11
2,dim_tienda,15,7
3,dim_promocion,29,7
4,dim_tiempo,731,10
5,fact_ventas,56193,14



=== Calidad: dim_cliente raw (5,000 filas) ===


,columna,tipo,nulos,pct_nulos,unicos
0,id_cliente,int64,0,0.0,5000
1,nombre,str,0,0.0,4995
2,sexo,str,0,0.0,2
3,fecha_nacimiento,str,0,0.0,4445
4,distrito,str,0,0.0,63
5,region,str,0,0.0,8
6,fecha_alta,str,0,0.0,112
7,segmento_programa,str,0,0.0,4



=== Calidad: dim_producto raw (506 filas) ===


...

,columna,tipo,nulos,pct_nulos,unicos
0,id_promocion,int64,0,0.00,29
1,nombre,str,0,0.00,15
2,tipo,str,0,0.00,5
3,descuento_pct,float64,0,0.00,7
4,fecha_inicio,str,0,0.00,29
5,fecha_fin,str,0,0.00,29
6,anio,float64,1,3.45,2



=== Calidad: dim_tiempo raw (731 filas) ===


,columna,tipo,nulos,pct_nulos,unicos
0,id_tiempo,int64,0,0.00,731
1,fecha,str,0,0.00,731
2,dia,int64,0,0.00,31
3,mes,int64,0,0.00,12
4,trimestre,int64,0,0.00,4
5,anio,int64,0,0.00,2
6,dia_semana,int64,0,0.00,7
7,es_fin_semana,bool,0,0.00,2
8,es_feriado,bool,0,0.00,2
9,nombre_feriado,str,699,95.62,16



=== Calidad: fact_ventas raw (56,193 filas) ===


,columna,tipo,nulos,pct_nulos,unicos
0,id_venta,int64,0,0.0,34755
1,numero_linea,int64,0,0.0,8
2,fecha,str,0,0.0,731
3,id_cliente,int64,0,0.0,5000
4,id_tienda,int64,0,0.0,15
5,id_producto,int64,0,0.0,506
6,id_promocion,int64,0,0.0,29
7,cantidad,int64,0,0.0,5
8,precio_unitario_lista,float64,0,0.0,110
9,descuento_pct,float64,451,0.8,7


#### Paso 4: Diagnosticar ruido esperado en la capa raw

In [37]:
def diagnosticar_ruido(tablas: dict[str, pd.DataFrame]) -> pd.DataFrame:
    fact = tablas["fact_ventas"]
    producto = tablas["dim_producto"]
    cliente = tablas["dim_cliente"]

    diagnostico = {
        "lineas_fact_ventas_raw": len(fact),
        "tickets_raw": fact["id_venta"].nunique() if "id_venta" in fact.columns else np.nan,
        "duplicados_grano_fact": (
            int(fact.duplicated(["id_venta", "numero_linea"]).sum())
            if {"id_venta", "numero_linea"}.issubset(fact.columns)
            else np.nan
        ),
        "nulos_descuento_pct": (
            int(fact["descuento_pct"].isna().sum())
            if "descuento_pct" in fact.columns
            else np.nan
        ),
        "nulos_marca_producto": (
            int(producto["marca"].isna().sum())
            if "marca" in producto.columns
            else np.nan
        ),
        "nombres_cliente_mayus": (
            int(cliente["nombre"].astype(str).str.isupper().sum())
            if "nombre" in cliente.columns
            else np.nan
        ),
        "nombres_cliente_minus": (
            int(cliente["nombre"].astype(str).str.islower().sum())
            if "nombre" in cliente.columns
            else np.nan
        ),
    }

    return pd.DataFrame(
        [{"indicador": k, "valor": v} for k, v in diagnostico.items()]
    )


diagnostico_ruido = diagnosticar_ruido(raw)
display(diagnostico_ruido)

print("Ejemplos de fechas raw:")
print(raw["fact_ventas"]["fecha"].head(12).tolist())

,indicador,valor
0,lineas_fact_ventas_raw,56193
1,tickets_raw,34755
2,duplicados_grano_fact,335
3,nulos_descuento_pct,451
4,nulos_marca_producto,7
5,nombres_cliente_mayus,750
6,nombres_cliente_minus,750


Ejemplos de fechas raw:
['01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024', '01/01/2024']


#### Paso 5: Funciones de limpieza

In [38]:
def parsear_fecha_mixta(serie: pd.Series) -> pd.Series:
    """
    Soporta fechas ISO YYYY-MM-DD y fechas latinas DD/MM/YYYY.
    Primero intenta ISO, luego formato latino y finalmente parseo general con dayfirst=True.
    """
    s = serie.astype("string").str.strip()

    fecha_iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    fecha_latam = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    fecha_general = pd.to_datetime(s, errors="coerce", dayfirst=True)

    return fecha_iso.fillna(fecha_latam).fillna(fecha_general)


def limpiar_texto(serie: pd.Series, title: bool = False) -> pd.Series:
    s = (
        serie
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA})

    if title:
        s = s.str.title()

    return s


def convertir_columnas_numericas(
    df: pd.DataFrame,
    columnas: list[str],
    enteras: list[str] | None = None,
) -> pd.DataFrame:
    df = df.copy()
    enteras = enteras or []

    for col in columnas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in enteras:
        if col in df.columns:
            df[col] = df[col].astype("Int64")

    return df


def convertir_booleano(serie: pd.Series) -> pd.Series:
    s = serie.astype("string").str.strip().str.lower()
    return s.isin({"true", "1", "si", "sí", "yes", "y"})

#### Paso 6: Estandarizar tipos, fechas y textos

In [39]:
etl = {nombre: df.copy() for nombre, df in raw.items()}

# -----------------------------
# Fechas
# -----------------------------
FECHAS_POR_TABLA = {
    "fact_ventas": ["fecha"],
    "dim_tiempo": ["fecha"],
    "dim_cliente": ["fecha_nacimiento", "fecha_alta"],
    "dim_promocion": ["fecha_inicio", "fecha_fin"],
}

fechas_invalidas = []

for nombre_tabla, columnas in FECHAS_POR_TABLA.items():
    df = etl[nombre_tabla].copy()

    for col in columnas:
        if col not in df.columns:
            continue

        original = df[col].copy()
        df[col] = parsear_fecha_mixta(df[col])

        invalidas = df[col].isna() & original.notna()
        if invalidas.any():
            fechas_invalidas.append({
                "tabla": nombre_tabla,
                "columna": col,
                "invalidas": int(invalidas.sum()),
                "ejemplos": original[invalidas].head(5).tolist(),
            })

    etl[nombre_tabla] = df

if fechas_invalidas:
    display(pd.DataFrame(fechas_invalidas))
    raise ValueError("Hay fechas no parseables. Revise los ejemplos mostrados.")

print("Fechas estandarizadas correctamente.")


# -----------------------------
# Numéricos
# -----------------------------
NUMERICAS = {
    "fact_ventas": [
        "id_venta",
        "numero_linea",
        "id_cliente",
        "id_tienda",
        "id_producto",
        "id_promocion",
        "cantidad",
        "precio_unitario_lista",
        "descuento_pct",
        "precio_unitario_final",
        "importe_venta",
        "costo_total",
        "margen",
    ],
    "dim_cliente": ["id_cliente"],
    "dim_producto": ["id_producto", "precio_lista", "costo_unitario_promedio"],
    "dim_tienda": ["id_tienda", "prob_seleccion"],
    "dim_promocion": ["id_promocion", "descuento_pct"],
    "dim_tiempo": ["id_tiempo", "dia", "mes", "trimestre", "anio", "dia_semana"],
}

ENTERAS = {
    "fact_ventas": [
        "id_venta",
        "numero_linea",
        "id_cliente",
        "id_tienda",
        "id_producto",
        "id_promocion",
        "cantidad",
    ],
    "dim_cliente": ["id_cliente"],
    "dim_producto": ["id_producto"],
    "dim_tienda": ["id_tienda"],
    "dim_promocion": ["id_promocion"],
    "dim_tiempo": ["id_tiempo", "dia", "mes", "trimestre", "anio", "dia_semana"],
}

for nombre_tabla, columnas in NUMERICAS.items():
    etl[nombre_tabla] = convertir_columnas_numericas(
        etl[nombre_tabla],
        columnas=columnas,
        enteras=ENTERAS.get(nombre_tabla, []),
    )

print("Columnas numéricas estandarizadas.")


# -----------------------------
# Textos
# -----------------------------
cliente = etl["dim_cliente"].copy()
for col in ["nombre", "sexo", "distrito", "region", "segmento_programa"]:
    if col in cliente.columns:
        cliente[col] = limpiar_texto(cliente[col], title=(col == "nombre"))
etl["dim_cliente"] = cliente

producto = etl["dim_producto"].copy()
for col in ["nombre", "categoria", "subcategoria", "marca", "asociacion_1", "asociacion_2", "asociacion_3"]:
    if col in producto.columns:
        producto[col] = limpiar_texto(producto[col], title=False)

if "categoria" in producto.columns:
    producto["categoria"] = producto["categoria"].str.title()

if "producto_estrella" in producto.columns:
    producto["producto_estrella"] = convertir_booleano(producto["producto_estrella"])

etl["dim_producto"] = producto

tienda = etl["dim_tienda"].copy()
for col in ["nombre", "canal", "region", "ciudad"]:
    if col in tienda.columns:
        tienda[col] = limpiar_texto(tienda[col], title=False)

if "canal" in tienda.columns:
    tienda["canal"] = tienda["canal"].str.title()

etl["dim_tienda"] = tienda

promocion = etl["dim_promocion"].copy()
for col in ["nombre", "tipo"]:
    if col in promocion.columns:
        promocion[col] = limpiar_texto(promocion[col], title=False)
etl["dim_promocion"] = promocion

print("Textos estandarizados.")

Fechas estandarizadas correctamente.
Columnas numéricas estandarizadas.
Textos estandarizados.


#### Paso 7: Imputar nulos controlados

In [40]:
# ------------------------------------------------------------
# 1. Recuperar descuento_pct desde dim_promocion
# ------------------------------------------------------------

fact = etl["fact_ventas"].copy()
promocion = etl["dim_promocion"].copy()

# Identificar únicamente las filas afectadas por el ruido.
mascara_descuento = fact["descuento_pct"].isna()

nulos_descuento_antes = int(
    mascara_descuento.sum()
)

# Relación entre cada promoción y su porcentaje original.
descuento_por_promocion = (
    promocion
    .set_index("id_promocion")["descuento_pct"]
)

# Recuperar el porcentaje usando la clave id_promocion.
fact.loc[
    mascara_descuento,
    "descuento_pct",
] = (
    fact.loc[
        mascara_descuento,
        "id_promocion",
    ]
    .map(descuento_por_promocion)
)


# ------------------------------------------------------------
# 2. Recalcular valores dependientes del descuento
# ------------------------------------------------------------

# El generador calcula primero el precio unitario final.
fact.loc[
    mascara_descuento,
    "precio_unitario_final",
] = (
    fact.loc[
        mascara_descuento,
        "precio_unitario_lista",
    ]
    * (
        1
        - fact.loc[
            mascara_descuento,
            "descuento_pct",
        ]
    )
).round(2)

# Después calcula el importe de la línea usando
# el precio unitario final ya redondeado.
fact.loc[
    mascara_descuento,
    "importe_venta",
] = (
    fact.loc[
        mascara_descuento,
        "precio_unitario_final",
    ]
    * fact.loc[
        mascara_descuento,
        "cantidad",
    ]
).round(2)

nulos_descuento_despues = int(
    fact["descuento_pct"].isna().sum()
)

etl["fact_ventas"] = fact


# ------------------------------------------------------------
# 3. Imputar las marcas eliminadas por el ruido
# ------------------------------------------------------------

producto = etl["dim_producto"].copy()

nulos_marca_antes = int(
    producto["marca"].isna().sum()
)

# La marca original no puede recuperarse desde otra dimensión.
producto["marca"] = producto["marca"].fillna(
    "Marca no identificada"
)

nulos_marca_despues = int(
    producto["marca"].isna().sum()
)

etl["dim_producto"] = producto


# ------------------------------------------------------------
# 4. Resumen de imputación
# ------------------------------------------------------------

resumen_imputacion = pd.DataFrame(
    [
        {
            "campo": "fact_ventas.descuento_pct",
            "nulos_antes": nulos_descuento_antes,
            "nulos_despues": nulos_descuento_despues,
            "tratamiento": (
                "Recuperado desde dim_promocion mediante id_promocion"
            ),
        },
        {
            "campo": "dim_producto.marca",
            "nulos_antes": nulos_marca_antes,
            "nulos_despues": nulos_marca_despues,
            "tratamiento": (
                "Imputado como 'Marca no identificada'"
            ),
        },
    ]
)

display(resumen_imputacion)

,campo,nulos_antes,nulos_despues,tratamiento
0,fact_ventas.descuento_pct,451,0,Recuperado desde dim_promocion mediante id_pro...
1,dim_producto.marca,7,0,Imputado como 'Marca no identificada'


#### Paso 8: Eliminar duplicados por grano de fact_ventas

In [41]:
fact = etl["fact_ventas"].copy()

duplicados_antes = contar_duplicados_grano(fact, ["id_venta", "numero_linea"])

fact["_orden_original"] = np.arange(len(fact))

fact = (
    fact
    .sort_values("_orden_original")
    .drop_duplicates(subset=["id_venta", "numero_linea"], keep="first")
    .drop(columns="_orden_original")
    .reset_index(drop=True)
)

duplicados_despues = contar_duplicados_grano(fact, ["id_venta", "numero_linea"])

etl["fact_ventas"] = fact

print("Duplicados por grano antes:", duplicados_antes)
print("Duplicados por grano después:", duplicados_despues)
print("Filas finales fact_ventas:", len(fact))

Duplicados por grano antes: 335
Duplicados por grano después: 0
Filas finales fact_ventas: 55858


#### Paso 9: Validar claves críticas e integridad referencial

In [42]:
fact = etl["fact_ventas"]

CLAVES_CRITICAS = {
    "dim_cliente": ["id_cliente"],
    "dim_producto": ["id_producto"],
    "dim_tienda": ["id_tienda"],
    "dim_promocion": ["id_promocion"],
    "dim_tiempo": ["fecha"],
    "fact_ventas": [
        "id_venta",
        "numero_linea",
        "fecha",
        "id_cliente",
        "id_tienda",
        "id_producto",
        "id_promocion",
    ],
}

errores_nulos_criticos = []

for nombre_tabla, columnas in CLAVES_CRITICAS.items():
    df = etl[nombre_tabla]

    for col in columnas:
        if col not in df.columns:
            errores_nulos_criticos.append({
                "tabla": nombre_tabla,
                "columna": col,
                "problema": "columna faltante",
                "cantidad": None,
            })
            continue

        nulos = int(df[col].isna().sum())

        if nulos > 0:
            errores_nulos_criticos.append({
                "tabla": nombre_tabla,
                "columna": col,
                "problema": "nulos críticos",
                "cantidad": nulos,
            })

if errores_nulos_criticos:
    display(pd.DataFrame(errores_nulos_criticos))
    raise ValueError("Hay claves críticas nulas o columnas faltantes.")

print("No hay nulos en claves críticas.")


fk_checks = [
    ("id_cliente", etl["dim_cliente"], "id_cliente", "fact_ventas → dim_cliente"),
    ("id_producto", etl["dim_producto"], "id_producto", "fact_ventas → dim_producto"),
    ("id_tienda", etl["dim_tienda"], "id_tienda", "fact_ventas → dim_tienda"),
    ("id_promocion", etl["dim_promocion"], "id_promocion", "fact_ventas → dim_promocion"),
    ("fecha", etl["dim_tiempo"], "fecha", "fact_ventas → dim_tiempo"),
]

fk_resultados = []

for col_fact, dim, col_dim, etiqueta in fk_checks:
    huerfanas = contar_huerfanas(fact[col_fact], dim[col_dim])

    fk_resultados.append({
        "relacion": etiqueta,
        "claves_huerfanas": huerfanas,
    })

    status = "OK" if huerfanas == 0 else "ERROR"
    print(f"{status} {etiqueta}: {huerfanas} huérfanas")

df_fk = pd.DataFrame(fk_resultados)
display(df_fk)

if (df_fk["claves_huerfanas"] > 0).any():
    raise ValueError("Hay claves foráneas huérfanas. Revise el ETL.")

No hay nulos en claves críticas.
OK fact_ventas → dim_cliente: 0 huérfanas
OK fact_ventas → dim_producto: 0 huérfanas
OK fact_ventas → dim_tienda: 0 huérfanas
OK fact_ventas → dim_promocion: 0 huérfanas
OK fact_ventas → dim_tiempo: 0 huérfanas


,relacion,claves_huerfanas
0,fact_ventas → dim_cliente,0
1,fact_ventas → dim_producto,0
2,fact_ventas → dim_tienda,0
3,fact_ventas → dim_promocion,0
4,fact_ventas → dim_tiempo,0


#### Paso 10: Validar cobertura de clientes y fecha_alta

In [43]:
cliente = etl["dim_cliente"].copy()
fact = etl["fact_ventas"].copy()

clientes_sin_venta = len(set(cliente["id_cliente"]) - set(fact["id_cliente"]))

primera_venta = (
    fact
    .groupby("id_cliente")["fecha"]
    .min()
    .rename("primera_venta")
)

cliente_validacion = (
    cliente
    .set_index("id_cliente")
    .join(primera_venta, how="left")
    .reset_index()
)

cliente_validacion["fecha_alta_coincide"] = (
    cliente_validacion["fecha_alta"] == cliente_validacion["primera_venta"]
)

fecha_alta_no_coincide = int((~cliente_validacion["fecha_alta_coincide"]).sum())

resumen_clientes = pd.DataFrame([
    {"validacion": "clientes", "valor": len(cliente)},
    {"validacion": "clientes_sin_venta", "valor": clientes_sin_venta},
    {"validacion": "fecha_alta_no_coincide_con_primera_venta", "valor": fecha_alta_no_coincide},
])

display(resumen_clientes)

if clientes_sin_venta > 0:
    display(
        cliente_validacion[
            cliente_validacion["primera_venta"].isna()
        ].head(20)
    )
    raise ValueError("Hay clientes sin venta asociada.")

if fecha_alta_no_coincide > 0:
    display(
        cliente_validacion[
            ~cliente_validacion["fecha_alta_coincide"]
        ][["id_cliente", "fecha_alta", "primera_venta"]].head(20)
    )
    raise ValueError("Hay clientes cuya fecha_alta no coincide con su primera venta.")

print("OK cobertura de clientes y fecha_alta.")

,validacion,valor
0,clientes,5000
1,clientes_sin_venta,0
2,fecha_alta_no_coincide_con_primera_venta,0


OK cobertura de clientes y fecha_alta.


#### Paso 11: Validar y reconstruir métricas derivadas

In [44]:
fact = etl["fact_ventas"].copy()

validaciones_base = pd.DataFrame([
    {
        "validacion": "cantidad <= 0",
        "filas": int((fact["cantidad"] <= 0).sum()),
    },
    {
        "validacion": "precio_unitario_lista < 0",
        "filas": int((fact["precio_unitario_lista"] < 0).sum()),
    },
    {
        "validacion": "precio_unitario_final < 0",
        "filas": int((fact["precio_unitario_final"] < 0).sum()),
    },
    {
        "validacion": "costo_total < 0",
        "filas": int((fact["costo_total"] < 0).sum()),
    },
    {
        "validacion": "importe_venta < 0",
        "filas": int((fact["importe_venta"] < 0).sum()),
    },
    {
        "validacion": "descuento_pct fuera de rango 0-1",
        "filas": int(((fact["descuento_pct"] < 0) | (fact["descuento_pct"] > 1)).sum()),
    },
])

display(validaciones_base)

if validaciones_base["filas"].sum() > 0:
    raise ValueError("Hay valores numéricos inválidos en fact_ventas.")


importe_calc_antes = (fact["cantidad"] * fact["precio_unitario_final"]).round(2)
margen_calc_antes = (fact["importe_venta"] - fact["costo_total"]).round(2)

diff_importe_antes = (fact["importe_venta"] - importe_calc_antes).abs()
diff_margen_antes = (fact["margen"] - margen_calc_antes).abs()

print("Diferencia máxima importe_venta antes:", diff_importe_antes.max())
print("Diferencia máxima margen antes:", diff_margen_antes.max())


# Reconstrucción determinística.
fact["importe_venta"] = (
    fact["cantidad"] * fact["precio_unitario_final"]
).round(2)

fact["margen"] = (
    fact["importe_venta"] - fact["costo_total"]
).round(2)


importe_calc_despues = (fact["cantidad"] * fact["precio_unitario_final"]).round(2)
margen_calc_despues = (fact["importe_venta"] - fact["costo_total"]).round(2)

diff_importe_despues = (fact["importe_venta"] - importe_calc_despues).abs()
diff_margen_despues = (fact["margen"] - margen_calc_despues).abs()

print("Diferencia máxima importe_venta después:", diff_importe_despues.max())
print("Diferencia máxima margen después:", diff_margen_despues.max())

assert diff_importe_despues.max() < 0.05, "importe_venta no cuadra con cantidad * precio final"
assert diff_margen_despues.max() < 0.05, "margen no cuadra con importe_venta - costo_total"

etl["fact_ventas"] = fact

print("\nOK métricas derivadas consistentes.")

,validacion,filas
0,cantidad <= 0,0
1,precio_unitario_lista < 0,0
2,precio_unitario_final < 0,0
3,costo_total < 0,0
4,importe_venta < 0,0
5,descuento_pct fuera de rango 0-1,0


Diferencia máxima importe_venta antes: 0.0
Diferencia máxima margen antes: 0.02999999999999936
Diferencia máxima importe_venta después: 0.0
Diferencia máxima margen después: 0.0

OK métricas derivadas consistentes.


#### Paso 12: Reporte de calidad después del ETL

In [45]:
calidad_despues = {}

for nombre, df in etl.items():
    calidad_despues[nombre] = reporte_calidad(df, f"{nombre} post-ETL")

comparativo = []

for nombre in TABLAS:
    comparativo.append({
        "tabla": nombre,
        "filas_raw": len(raw[nombre]),
        "filas_post_etl": len(etl[nombre]),
        "nulos_raw": int(calidad_antes[nombre]["nulos"].sum()),
        "nulos_post_etl": int(calidad_despues[nombre]["nulos"].sum()),
    })

df_comparativo = pd.DataFrame(comparativo)
display(df_comparativo)


=== Calidad: dim_cliente post-ETL (5,000 filas) ===


,columna,tipo,nulos,pct_nulos,unicos
0,id_cliente,Int64,0,0.0,5000
1,nombre,string,0,0.0,4993
2,sexo,string,0,0.0,2
3,fecha_nacimiento,datetime64[us],0,0.0,4445
4,distrito,string,0,0.0,63
5,region,string,0,0.0,8
6,fecha_alta,datetime64[us],0,0.0,112
7,segmento_programa,string,0,0.0,4



=== Calidad: dim_producto post-ETL (506 filas) ===


,columna,tipo,nulos,pct_nulos,unicos
0,id_producto,Int64,0,0.00,506
1,nombre,string,0,0.00,502
2,categoria,string,0,0.00,12
3,subcategoria,string,0,0.00,54
4,marca,string,0,0.00,180
5,precio_lista,float64,0,0.00,110
6,costo_unitario_promedio,float64,0,0.00,211
7,producto_estrella,bool,0,0.00,2
8,asociacion_1,string,0,0.00,34
9,asociacion_2,string,25,4.94,32



=== Calidad: dim_tienda post-ETL (15 filas) ===


...


=== Calidad: dim_tiempo post-ETL (731 filas) ===


,columna,tipo,nulos,pct_nulos,unicos
0,id_tiempo,Int64,0,0.00,731
1,fecha,datetime64[us],0,0.00,731
2,dia,Int64,0,0.00,31
3,mes,Int64,0,0.00,12
4,trimestre,Int64,0,0.00,4
5,anio,Int64,0,0.00,2
6,dia_semana,Int64,0,0.00,7
7,es_fin_semana,bool,0,0.00,2
8,es_feriado,bool,0,0.00,2
9,nombre_feriado,str,699,95.62,16



=== Calidad: fact_ventas post-ETL (55,858 filas) ===


,columna,tipo,nulos,pct_nulos,unicos
0,id_venta,Int64,0,0.0,34755
1,numero_linea,Int64,0,0.0,8
2,fecha,datetime64[us],0,0.0,731
3,id_cliente,Int64,0,0.0,5000
4,id_tienda,Int64,0,0.0,15
5,id_producto,Int64,0,0.0,506
6,id_promocion,Int64,0,0.0,29
7,cantidad,Int64,0,0.0,5
8,precio_unitario_lista,float64,0,0.0,110
9,descuento_pct,float64,0,0.0,7


,tabla,filas_raw,filas_post_etl,nulos_raw,nulos_post_etl
0,dim_cliente,5000,5000,0,0
1,dim_producto,506,506,135,128
2,dim_tienda,15,15,0,0
3,dim_promocion,29,29,1,1
4,dim_tiempo,731,731,699,699
5,fact_ventas,56193,55858,451,0


#### Paso 13: Descripción del esquema estrella

El datamart queda organizado bajo un esquema estrella. La tabla central es `fact_ventas`, donde cada registro representa una línea de venta identificada por la combinación `id_venta` + `numero_linea`.

A partir de esa tabla de hechos se conectan las dimensiones principales del negocio:

- `dim_cliente`, mediante `id_cliente`, permite analizar las ventas por cliente, región, distrito, sexo y segmento de fidelización.
- `dim_producto`, mediante `id_producto`, permite analizar las ventas por producto, categoría, subcategoría, marca y condición de producto estrella.
- `dim_tienda`, mediante `id_tienda`, permite analizar el comportamiento por tienda, canal, ciudad y región de operación.
- `dim_promocion`, mediante `id_promocion`, permite identificar qué promociones participaron en cada venta y evaluar su impacto.
- `dim_tiempo`, mediante `fecha`, permite agrupar las ventas por día, mes, trimestre, año, fines de semana y feriados.

En términos prácticos, `fact_ventas` concentra las métricas cuantitativas del modelo: `cantidad`, `precio_unitario_lista`, `descuento_pct`, `precio_unitario_final`, `importe_venta`, `costo_total` y `margen`. Las dimensiones aportan el contexto necesario para segmentar y explicar esas métricas.

#### Paso 14: Construir diccionario de datos

In [46]:
from scripts.generar_diccionario_datos import (
    generar_diccionario_datos,
)

ruta_diccionario = DOCS_DIR / "diccionario_datos.md"

df_diccionario = generar_diccionario_datos(
    tablas=etl,
    ruta_salida=ruta_diccionario,
)

print(df_diccionario.head(5))
print()

print(
    f"Total campos documentados: {len(df_diccionario)}"
)

print(
    "Diccionario exportado a:",
    ruta_relativa(ruta_diccionario),
)

         tabla             campo  tipo             descripcion es_clave_fk
0  dim_cliente        id_cliente   int          PK del cliente          Sí
1  dim_cliente            nombre   str      Nombre del cliente          No
2  dim_cliente              sexo   str        Sexo del cliente          No
3  dim_cliente  fecha_nacimiento  date     Fecha de nacimiento          No
4  dim_cliente          distrito   str  Distrito de residencia          No

Total campos documentados: 57
Diccionario exportado a: docs\diccionario_datos.md


#### Paso 15: Exportar tablas limpias a data/processed

In [47]:
def preparar_para_exportar(nombre_tabla: str, df: pd.DataFrame) -> pd.DataFrame:
    df_export = df.copy()

    columnas_fecha = FECHAS_POR_TABLA.get(nombre_tabla, [])

    for col in columnas_fecha:
        if col in df_export.columns and pd.api.types.is_datetime64_any_dtype(df_export[col]):
            df_export[col] = df_export[col].dt.strftime("%Y-%m-%d")

    return df_export


for nombre, df in etl.items():
    ruta_salida = DATA_PROCESSED / TABLAS[nombre]
    df_export = preparar_para_exportar(nombre, df)

    df_export.to_csv(ruta_salida, sep=SEP, index=False, encoding=ENC)

    print(f"Exportado {ruta_relativa(ruta_salida)}: {len(df_export):,} filas")


fact_final = etl["fact_ventas"]

print("\n=== RESUMEN FINAL DEL DATAMART ===")
print(f"Clientes: {etl['dim_cliente']['id_cliente'].nunique():,}")
print(f"Productos: {etl['dim_producto']['id_producto'].nunique():,}")
print(f"Tiendas: {etl['dim_tienda']['id_tienda'].nunique():,}")
print(f"Promociones: {etl['dim_promocion']['id_promocion'].nunique():,}")
print(f"Días calendario: {len(etl['dim_tiempo']):,}")
print(f"Tickets: {fact_final['id_venta'].nunique():,}")
print(f"Líneas: {len(fact_final):,}")
print(f"Ventas totales: S/ {fact_final['importe_venta'].sum():,.2f}")
print(f"Margen total: S/ {fact_final['margen'].sum():,.2f}")
print(f"Periodo: {fact_final['fecha'].min().date()} a {fact_final['fecha'].max().date()}")
print("\nDatamart listo para importar en Power BI desde data/processed/.")

Exportado data\processed\dim_cliente.csv: 5,000 filas
Exportado data\processed\dim_producto.csv: 506 filas
Exportado data\processed\dim_tienda.csv: 15 filas
Exportado data\processed\dim_promocion.csv: 29 filas
Exportado data\processed\dim_tiempo.csv: 731 filas
Exportado data\processed\fact_ventas.csv: 55,858 filas

=== RESUMEN FINAL DEL DATAMART ===
Clientes: 5,000
Productos: 506
Tiendas: 15
Promociones: 29
Días calendario: 731
Tickets: 34,755
Líneas: 55,858
Ventas totales: S/ 1,428,828.50
Margen total: S/ 392,905.10
Periodo: 2024-01-01 a 2025-12-31

Datamart listo para importar en Power BI desde data/processed/.


#### Paso 16: Verificación rápida de archivos exportados

In [48]:
verificacion_archivos = []

for nombre, archivo in TABLAS.items():
    ruta = DATA_PROCESSED / archivo

    verificacion_archivos.append({
        "tabla": nombre,
        "archivo": ruta_relativa(ruta),
        "existe": ruta.exists(),
        "peso_kb": round(ruta.stat().st_size / 1024, 2) if ruta.exists() else None,
    })

display(pd.DataFrame(verificacion_archivos))

,tabla,archivo,existe,peso_kb
0,dim_cliente,data\processed\dim_cliente.csv,True,398.68
1,dim_producto,data\processed\dim_producto.csv,True,56.30
2,dim_tienda,data\processed\dim_tienda.csv,True,0.80
3,dim_promocion,data\processed\dim_promocion.csv,True,1.94
4,dim_tiempo,data\processed\dim_tiempo.csv,True,34.72
5,fact_ventas,data\processed\fact_ventas.csv,True,3322.87


### Conclusiones — Parte 1

- El ETL consume la capa `data/raw` producida por el generador centralizado.
- Se corrigieron fechas mixtas, nulos controlados, duplicados de grano y ruido textual.
- El datamart final conserva el esquema estrella con `fact_ventas` en el centro.
- La tabla de hechos mantiene el grano `id_venta + numero_linea`.
- Las métricas `importe_venta` y `margen` quedan reconstruidas y validadas.
- La marca de productos no identificable se clasifica como `Marca no identificada`, evitando inventar información maestra.
- Las tablas finales fueron exportadas a `data/processed`.
- El diccionario de datos fue exportado a `docs/diccionario_datos.md`.

Siguiente paso: ejecutar los notebooks analíticos sobre la capa `data/processed`.